# BuildArena S02 - Human AI Engineering Challenge

Working notebook for preparing a submission: setup checklist, run configuration, engineering-system prompt, trajectory analysis, and a Kaggle writeup template.

**Important:** this notebook contains no prebuilt machine and no ready-made controller. Those are prohibited. The machine and controller must be produced by your agent through the BuildArena 2.0 MCP.

## 1. Requirements checklist

Edit the values to `True` as you complete each item.

In [ ]:
checklist = {
    'Besiege base game (Steam key received via form/email)': False,
    'The Broken Beyond DLC installed': False,
    'Official BuildArena 2.0 MCP configured': False,
    'BuildArena MOD installed (produces CSV trajectory)': False,
    'Machine built through BuildArena 2.0 MCP (or human-built for Human Boss Challenge)': False,
    'One run with CSV trajectory file saved': False,
    'Kaggle writeup drafted': False,
    'Video recorded and uploaded (YouTube or other public link) - strongly encouraged': False,
}

for item, done in checklist.items():
    print(('[x] ' if done else '[ ] ') + item)
print()
print(sum(checklist.values()), 'of', len(checklist), 'done')

## 2. Mode and rules

| Mode | Human guidance after Engineering Start | Score coefficient |
|---|---|---|
| Autopilot | None | x1.25 |
| Copilot | Natural-language collaboration allowed | x1.00 |

In both modes, humans may **not**:
- manually edit the machine or controller
- directly tune engineering parameters
- pilot development or final flights

Prebuilt mission machines and ready-to-run mission controllers are prohibited. Prepare the engineering system, not the finished answer.

In [ ]:
# Run configuration - fill in before Engineering Start
MODE = 'Autopilot'   # 'Autopilot' or 'Copilot'
COEFFICIENTS = {'Autopilot': 1.25, 'Copilot': 1.00}

run_config = {
    'mode': MODE,
    'score_coefficient': COEFFICIENTS[MODE],
    'model': 'TODO: model name',
    'harness': 'TODO: agent framework / harness',
    'tools': ['BuildArena 2.0 MCP'],
    'skills': [],
    'engineering_start_time': 'TODO: timestamp',
}

attestation = {
    'no_manual_machine_or_controller_edits': True,
    'no_direct_parameter_tuning_by_human': True,
    'no_human_piloting': True,
    'no_prebuilt_machine_or_controller': True,
    'no_human_guidance_after_start (Autopilot only)': MODE == 'Autopilot',
}

import json
print(json.dumps(run_config, indent=2))
print(json.dumps(attestation, indent=2))

## 3. Engineering-system prompt (Autopilot)

In Autopilot you configure the model, harness, tools, skills, system prompt and initial instruction **before** Engineering Start. There is no length limit. Keep the prompt general (how to engineer), not a pre-solved design.

In [ ]:
SYSTEM_PROMPT = '''You are an autonomous engineer working inside Besiege through the BuildArena 2.0 MCP.

Operating loop:
1. Read the mission specification and scoring rules from the MCP before designing anything.
2. Propose a design hypothesis and the reason you expect it to work.
3. Build the machine only through MCP tools. Never assume a block is placed; verify via MCP state queries.
4. Write a controller, run a test flight, and read the resulting trajectory data.
5. Diagnose failures (instability, structural break, insufficient thrust, control lag) from the data, then change ONE major thing at a time.
6. Keep a log of every iteration: change made, expected effect, observed effect.
7. Stop iterating when the budget is nearly used; make sure the best verified version is the one submitted.

Rules: no human will edit or pilot anything. Make every engineering decision yourself.
'''

INITIAL_INSTRUCTION = '''Begin the engineering run. Complete the mission with a machine and controller you design yourself. Record your reasoning and iteration log as you go.'''

with open('system_prompt.txt', 'w') as f:
    f.write(SYSTEM_PROMPT)
with open('initial_instruction.txt', 'w') as f:
    f.write(INITIAL_INSTRUCTION)

print('Saved system_prompt.txt and initial_instruction.txt')
print('System prompt length:', len(SYSTEM_PROMPT), 'chars')

## 4. Trajectory analysis

Point `CSV_PATH` at the trajectory file generated by the BuildArena MOD. The column names are not assumed: the loader inspects the file and plots every numeric column against the first numeric column (usually time).

In [ ]:
import os
import pandas as pd

CSV_PATH = 'trajectory.csv'   # TODO: path to your BuildArena MOD CSV

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print('Shape:', df.shape)
    print('Columns:', list(df.columns))
    display(df.head())
    display(df.describe())
else:
    df = None
    print('No file at', CSV_PATH, '- set CSV_PATH after your run.')

In [ ]:
if df is not None:
    import matplotlib.pyplot as plt

    num = df.select_dtypes('number')
    if num.shape[1] >= 2:
        xcol = num.columns[0]
        ycols = list(num.columns[1:])
        ncols = 3
        nrows = (len(ycols) + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows), squeeze=False)
        for ax, col in zip(axes.flat, ycols):
            ax.plot(num[xcol], num[col])
            ax.set_title(col)
            ax.set_xlabel(xcol)
            ax.grid(alpha=0.3)
        for ax in list(axes.flat)[len(ycols):]:
            ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print('Not enough numeric columns to plot.')

## 5. Iteration log

Record what the agent tried. This feeds directly into the writeup. The entries should reflect what the agent did, not human edits.

In [ ]:
import pandas as pd

iteration_log = pd.DataFrame(columns=['iteration', 'change', 'expected_effect', 'observed_effect', 'outcome'])

def log_iteration(change, expected, observed, outcome):
    global iteration_log
    iteration_log.loc[len(iteration_log)] = [len(iteration_log) + 1, change, expected, observed, outcome]

# Example (delete): log_iteration('Added a second thruster', 'More lift', 'Wobble increased', 'worse')
iteration_log

## 6. Kaggle writeup template

Generates `writeup.md` from the fields below. Fill them in, run the cell, and paste the result into Kaggle.

In [ ]:
VIDEO_URL = 'TODO: public video link (YouTube preferred)'
WHAT_I_BUILT = 'TODO: one paragraph describing the machine and what it does'
HOW_I_BUILT_IT = 'TODO: model, harness, tools, skills, prompt strategy, how the agent iterated'
FAILURES = 'TODO: what failed, wobbled, exploded, and how the agent recovered'
RESULT = 'TODO: outcome of the submitted run and what the CSV shows'

writeup = f'''# BuildArena S02 Submission

**Mode:** {run_config['mode']} (coefficient x{run_config['score_coefficient']})
**Model / harness:** {run_config['model']} / {run_config['harness']}
**Video:** {VIDEO_URL}

## What I built
{WHAT_I_BUILT}

## How I built it
{HOW_I_BUILT_IT}

## Iteration highlights
{iteration_log.to_markdown(index=False) if len(iteration_log) else 'TODO: add iterations'}

## What went wrong
{FAILURES}

## Result
{RESULT}

## Compliance
No manual edits to the machine or controller, no direct human parameter tuning, no human piloting, and no prebuilt machine or controller were used.
'''

with open('writeup.md', 'w') as f:
    f.write(writeup)
print(writeup)